In [1]:
import torch 
from torchvision import datasets, transforms
from torch.utils.data import DataLoader,Subset
import torch.nn as nn 

In [2]:
class ResidualBlock(nn.Module):

    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False
        )

        self.bn1 = nn.BatchNorm2d(out_channels)

        self.conv2 = nn.Conv2d(
            out_channels,
            out_channels,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False
        )

        self.bn2 = nn.BatchNorm2d(out_channels)

        self.relu = nn.ReLU(inplace=True)

        if in_channels != out_channels or stride != 1:
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_channels,
                    out_channels,
                    kernel_size=1,
                    stride=stride,
                    bias=False
                ),
                nn.BatchNorm2d(out_channels)
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):

        identity = self.shortcut(x)

        # Main path
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        # Residual connection
        out = out + identity

        out = self.relu(out)

        return out

In [3]:
class ResNet(nn.Module):

    def __init__(self):
        super().__init__()

        # Initial convolution
        self.conv = nn.Conv2d(
            3,
            64,
            kernel_size=3,
            stride=1,
            padding=1,
            bias=False
        )

        self.bn = nn.BatchNorm2d(64)

        self.relu = nn.ReLU(inplace=True)

        # Residual layers
        self.layer1 = nn.Sequential(
            ResidualBlock(64, 64),
            ResidualBlock(64, 64)
        )

        self.layer2 = nn.Sequential(
            ResidualBlock(64, 128, stride=2),
            ResidualBlock(128, 128)
        )

        self.layer3 = nn.Sequential(
            ResidualBlock(128, 256, stride=2),
            ResidualBlock(256, 256)
        )

        # Classification
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

        self.fc = nn.Linear(256, 10)

    def forward(self, x):

        # Initial convolution
        x = self.conv(x)
        x = self.bn(x)
        x = self.relu(x)

        # Residual layers
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)

        # Global Average Pooling
        x = self.avgpool(x)

        # Flatten batch dimension
        x = torch.flatten(x, 1)

        # Classification
        x = self.fc(x)

        return x

In [4]:

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5, 0.5, 0.5),
        std=(0.5, 0.5, 0.5)
    )
])

train_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=False,
    transform=transform
)

test_dataset = datasets.CIFAR10(
    root="./data",
    train=False,
    download=False,
    transform=transform
)

# Take only 1,000 training images
train_dataset = Subset(train_dataset, range(1000))

# Take only 1,000 test images
test_dataset = Subset(test_dataset, range(1000))

train_loader = DataLoader(
    train_dataset,
    batch_size=128,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False
)

In [5]:
model = ResNet()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

epochs = 10

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for X, y in train_loader:
        # Forward
        logits = model(X)
        # Loss
        loss = criterion(logits, y)
        # Clear old gradients
        optimizer.zero_grad()
        # Backpropagation
        loss.backward()
        # Update parameters
        optimizer.step()
        
        running_loss += loss.item()

    print(
        f"Epoch [{epoch+1}/{epochs}], "
        f"Loss: {running_loss / len(train_loader):.4f}"
    )

Epoch [1/10], Loss: 2.2283
Epoch [2/10], Loss: 1.8482
Epoch [3/10], Loss: 1.6922
Epoch [4/10], Loss: 1.5668
Epoch [5/10], Loss: 1.4867
Epoch [6/10], Loss: 1.4018
Epoch [7/10], Loss: 1.2748
Epoch [8/10], Loss: 1.2026
Epoch [9/10], Loss: 1.1152
Epoch [10/10], Loss: 1.0670


In [6]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for X, y in test_loader:

        logits = model(X)

        predictions = torch.argmax(logits, dim=1)

        total += y.size(0)

        correct += (predictions == y).sum().item()

accuracy = correct / total

print("Test Accuracy:", accuracy)

Test Accuracy: 0.379
